<a href="https://colab.research.google.com/github/sheng13/Ai--/blob/main/%E3%80%8C0702_Colab_LINE_Bot_with_GEMINI_Stateful_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [43]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [44]:
import os
from pyngrok import ngrok

In [45]:
ngrok.kill()

In [46]:
import requests
from pyngrok import ngrok

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url   # ⭐ 不要加 /callback

print(f"Ngrok URL: {webhook_url}")

def update_line_webhook(webhook_url):
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已更新：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

update_line_webhook(webhook_url)


Ngrok URL: https://alla-mammillate-tawanna.ngrok-free.dev
✅ LINE Webhook URL 已更新：https://alla-mammillate-tawanna.ngrok-free.dev


True

In [47]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [48]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [49]:
result = stateful_query("iphone歷史")
print(result)

iPhone 的歷史是一段充滿創新、顛覆和持續演進的旅程，它重新定義了手機，並對全球科技和文化產生了深遠的影響。以下是 iPhone 的主要歷史里程碑和關鍵演變：

**一、 誕生與初代 (2007)**

*   **2007年1月9日：** 史蒂夫·賈伯斯在 Macworld 大會上首次發布 iPhone，將其描述為「一個 iPod、一個電話、一個網路通訊器」的三合一設備。
*   **2007年6月29日：** 初代 iPhone (俗稱 iPhone 2G) 在美國上市。
*   **關鍵創新：**
    *   多點觸控電容螢幕（告別實體鍵盤）。
    *   創新的使用者介面 (UI) 和手勢操作。
    *   Safari 瀏覽器提供完整的網路體驗。
    *   視覺語音信箱。
    *   基於 OS X 的作業系統 (當時稱為 iPhone OS)。
*   **限制：** 僅支援 2G 網路、沒有 App Store、不能複製貼上、不能錄影。

**二、 App Store 的崛起與 3G 時代 (2008-2009)**

*   **iPhone 3G (2008)：**
    *   支援 3G 網路，大幅提升上網速度。
    *   **引入 App Store！** 這是 iPhone 歷史上最重要的里程碑之一，開創了行動應用程式經濟，讓第三方開發者能為 iPhone 創造無限可能。
    *   內建 GPS 定位。
    *   塑膠背蓋，價格更親民。
*   **iPhone 3GS (2009)：**
    *   「S」代表 Speed (速度)，處理器、記憶體和圖形處理能力顯著提升。
    *   首次加入視訊錄影功能和自動對焦的 320 萬畫素相機。
    *   支援語音控制 (Voice Control)。
    *   終於支援複製、剪下、貼上功能。

**三、 設計革新與 Retina 顯示器 (2010-2011)**

*   **iPhone 4 (2010)：**
    *   **重大設計革新：** 玻璃背面和不鏽鋼邊框，更方正、時尚的外觀。
    *   **Retina 顯示器：** 首次引入高解析度螢幕，像素密度高達 326 ppi，顯示效果極為細膩。
    * 

In [50]:
result2 = stateful_query("iphone創始人")
print(result2)

iPhone 的創始人是 **史蒂夫·賈伯斯 (Steve Jobs)**。

雖然蘋果公司是一個龐大的團隊，有許多工程師、設計師和管理者共同努力才打造出 iPhone，但正是賈伯斯的願景、領導力以及他對極致使用者體驗和簡潔設計的執著，才催生了第一代 iPhone，並引導其走向成功。

他於 2007 年在 Macworld 大會上親自發布了 iPhone，並將其定位為「一個 iPod、一個電話、一個網路通訊器」的三合一革命性產品。他的演講和對產品的介紹，至今仍被視為科技產品發布的經典案例。


In [ ]:
from flask import Flask, request

from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # ✅ 改 1：安全取得 signature（沒有也不會炸）
    signature = request.headers.get('X-Line-Signature', '')

    body = request.get_data(as_text=True)
    print("BODY:", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        # ✅ 改 2：Verify webhook 時不要 abort
        print("Invalid signature (ignored for webhook verify)")
        return 'OK', 200

    # ✅ 改 3：明確回 200
    return 'OK', 200


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
        else:
            reply_text = event.message.text

        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=reply_text)]
            )
        )


if __name__ == "__main__":
    app.run(port=port)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 22:58:29] "POST / HTTP/1.1" 200 -


BODY: {"destination":"U287a3d06a20703e880b3c1fda99f4939","events":[]}
Invalid signature (ignored for webhook verify)


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 22:58:45] "POST / HTTP/1.1" 200 -


BODY: {"destination":"U287a3d06a20703e880b3c1fda99f4939","events":[{"type":"message","message":{"type":"text","id":"595583485512253731","quoteToken":"liTrGZj05WcKpbtrB3kryvzgrVkzyO3z2Vv9g8imz5fIDPTlEwMEt98ZGHLT13Cmcrv_4LnZ5I7Vn0h-0i54VtAZn6XTIxsPje_kT8xEsTLDwnZHSrc8bZw1zM48r3UYUJTnLRoVaecrBnEcgeJI2A","markAsReadToken":"5F-3OYtqzloX_lheoO6aTvZid9c8QK4juJpaFl0DroxZmX89IuHHpyxUXpkLSpl4cNz8pHpo6nWAndlBzkiw27a6p1cTI19tMsTRk0hwfBdz5iVFZhDU6WaJuyh4Ea4QjA6ib5vf6PSrFSgkDDSJgLsyjmBZcvrpN8sJZOocE1B1p4tJFqSOJkkhDLlR0urQueBXc7YH6F4eFkrglrrkTQ","text":"AIiphone歷史"},"webhookEventId":"01KEDAY6GS1GF7JZ3EH5CJXEYV","deliveryContext":{"isRedelivery":false},"timestamp":1767826724893,"source":{"type":"user","userId":"Ubffffa3746b3dc784a8659f16c5692d8"},"replyToken":"5c6832a7907b4c9e9867dc93ece18dca","mode":"active"}]}
Invalid signature (ignored for webhook verify)
